# Grants merge data-quality check
Run from the repository root. This profiles source inventory grain and timing gaps; field completeness does not establish current eligibility. Bulk data remains a research inventory.


In [1]:
import json, collections, hashlib
from pathlib import Path
path = Path("output/grants-db/grants-active.jsonl")
rows = [json.loads(line) for line in path.open()]
audit = {
    "records": len(rows),
    "duplicate_ids": len(rows) - len({r["id"] for r in rows}),
    "sources": dict(collections.Counter(r["source"] for r in rows)),
    "complete_by_type": dict(collections.Counter(r["recordType"] for r in rows if r["readiness"]["status"] == "ready")),
    "foundation_rows_without_exact_deadline": sum(r["recordType"] == "foundation" and not r["window"].get("deadline") for r in rows),
    "dated_rows_without_deadline_time": sum(bool(r["window"].get("deadline")) and not r["window"].get("deadlineTime") for r in rows),
    "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
}
audit


{
  "records": 30263,
  "duplicate_ids": 0,
  "sources": {
    "curated": 80,
    "grants.gov": 1244,
    "irs_990pf": 24530,
    "sam.gov": 2675,
    "illinois_nofo": 126,
    "illinois_csfa": 1608
  },
  "complete_by_type": {
    "opportunity": 458,
    "foundation": 17637,
    "standing_program": 228
  },
  "foundation_rows_without_exact_deadline": 24530,
  "dated_rows_without_deadline_time": 1523,
  "sha256": "b0c11b5fd003f5956a9cc1002361b4ea26e60f751fbacb3206e9fb5e81d4a9c7"
}

All foundation records are treated as application-window unconfirmed. Standing program listings do not assert an open round. Date-only deadlines are displayed without inventing a time or timezone. Source URLs are validated before rendering links. Staff promotion remains unverified and preserves the full source row.
The import loader re-read `grants_active`: 30,263 rows, 18,323 source-complete records. SQL counts are also tested through the staff API/browser; tests cover paging, role filters, source-change invalidation and idempotent promotion.
